# ML 07 — End-to-end Phase 1 evaluation and model comparison

This notebook retrains the baseline and tree models, generates fresh out-of-fold predictions, compares family-held-out, family leave-one-group-out and layout-held-out protocols, evaluates an engineered-feature ablation, estimates family-clustered uncertainty, and displays diagnostics inline. No saved result CSV is used as an input.

In [ ]:
from pathlib import Path
import os

# Works whether Jupyter starts in the repository root or this notebook's folder.
_candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in _candidates if (p / "data" / "raw").exists() and (p / "Machine Learning Pipeline").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Could not find the repository root. Open this notebook from inside the T006 project folder.")
DATA_RAW = ROOT / "data" / "raw"
DATA_PROCESSED = ROOT / "data" / "processed"
RESULTS = ROOT / "results"
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
RESULTS.mkdir(parents=True, exist_ok=True)
print("Project root:", ROOT)

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

processed_path = DATA_PROCESSED / "feature_engineered_dataset.csv"
if not processed_path.exists():
    raise FileNotFoundError("Run ML_01, then ML_03 first; ML_03 builds the engineered modelling dataset from raw data.")
df = pd.read_csv(processed_path)
TARGET = "target_U_canopy_mean_m_s"
NUMERIC_FEATURES = ["plan_area_density", "plan_area_density_sq", "height_std_m", "height_max_m", "height_min_m", "height_range_m", "height_skew", "frac_taller_than_mean", "wind_direction_sin", "wind_direction_cos", "height_coefficient_variation"]
CATEGORICAL_FEATURES = ["configuration", "horizontal_configuration", "density_class"]
FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES
BASELINE_NUMERIC = ["plan_area_density", "height_std_m", "height_max_m", "building_count", "wind_direction_sin", "wind_direction_cos"]
BASELINE_FEATURES = BASELINE_NUMERIC + CATEGORICAL_FEATURES
print("Dataset shape:", df.shape)
print("Target:", TARGET)

## 1. Define metrics, preprocessing and models

MAE is the primary easily interpreted error in m/s; RMSE emphasizes large misses; normalized RMSE scales RMSE by the observed target range; R² compares predictions with a constant-mean reference. A training-fold mean baseline is included to verify that the features add predictive value. All preprocessing is fitted within each training fold.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupKFold, LeaveOneGroupOut
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from xgboost import XGBRegressor


def make_models(numeric_features=NUMERIC_FEATURES, categorical_features=CATEGORICAL_FEATURES):
    linear_pre = ColumnTransformer([
        ("numeric", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), numeric_features),
        ("categorical", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(handle_unknown="ignore"))]), categorical_features),
    ])
    tree_pre = ColumnTransformer([
        ("numeric", SimpleImputer(strategy="median"), numeric_features),
        ("categorical", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(handle_unknown="ignore"))]), categorical_features),
    ])
    return {
        "Mean baseline": None,
        "Ridge Regression": Pipeline([("preprocess", linear_pre), ("model", Ridge(alpha=1.0))]),
        "Random Forest": Pipeline([("preprocess", tree_pre), ("model", RandomForestRegressor(n_estimators=100, min_samples_leaf=2, max_features=0.9, n_jobs=-1, random_state=42))]),
        "XGBoost": Pipeline([("preprocess", tree_pre), ("model", XGBRegressor(n_estimators=100, max_depth=2, learning_rate=0.03, min_child_weight=3, subsample=0.85, colsample_bytree=0.9, reg_lambda=2.0, objective="reg:squarederror", tree_method="hist", n_jobs=4, random_state=42))]),
    }

PROTOCOLS = {
    "family_5fold": (GroupKFold(n_splits=5), "family_id"),
    "family_loo": (LeaveOneGroupOut(), "family_id"),
    "layout_5fold": (GroupKFold(n_splits=5), "layout_id"),
}
print("Models:", list(make_models()))
print("Validation protocols:", list(PROTOCOLS))

## 2. Run the cross-validation experiments

In [ ]:
def run_cv(protocol_name, numeric_features=NUMERIC_FEATURES, categorical_features=CATEGORICAL_FEATURES, feature_set="engineered"):
    splitter, group_column = PROTOCOLS[protocol_name]
    feature_columns = numeric_features + categorical_features
    X_local = df[feature_columns]
    y_local = df[TARGET]
    groups_local = df[group_column]
    prediction_rows, fold_rows, importance_rows = [], [], []
    for fold, (train_idx, test_idx) in enumerate(splitter.split(X_local, y_local, groups_local), start=1):
        train_groups = set(groups_local.iloc[train_idx])
        test_groups = set(groups_local.iloc[test_idx])
        assert train_groups.isdisjoint(test_groups)
        if group_column == "family_id":
            assert set(df.iloc[train_idx].family_id).isdisjoint(set(df.iloc[test_idx].family_id))
        baseline_value = float(y_local.iloc[train_idx].mean())
        for model_name, model in make_models(numeric_features, categorical_features).items():
            if model is None:
                predictions = np.repeat(baseline_value, len(test_idx))
            else:
                model.fit(X_local.iloc[train_idx], y_local.iloc[train_idx])
                predictions = model.predict(X_local.iloc[test_idx])
                if model_name in {"Random Forest", "XGBoost"} and protocol_name == "family_5fold" and feature_set == "engineered":
                    feature_names = model.named_steps["preprocess"].get_feature_names_out()
                    for name, importance in zip(feature_names, model.named_steps["model"].feature_importances_):
                        raw_name = name.split("__", 1)[-1]
                        original = next((c for c in categorical_features if raw_name == c or raw_name.startswith(c + "_")), raw_name)
                        importance_rows.append({"protocol": protocol_name, "fold": fold, "model": model_name, "feature": original, "importance": float(importance)})
            actual = y_local.iloc[test_idx].to_numpy()
            fold_rows.append({"protocol": protocol_name, "feature_set": feature_set, "fold": fold, "model": model_name,
                              "n_test_cases": len(test_idx), "n_test_groups": len(test_groups),
                              "mae_m_s": mean_absolute_error(actual, predictions),
                              "rmse_m_s": np.sqrt(mean_squared_error(actual, predictions))})
            for idx, observed, predicted in zip(test_idx, actual, predictions):
                row = df.iloc[idx]
                prediction_rows.append({"protocol": protocol_name, "feature_set": feature_set, "fold": fold,
                                        "model": model_name, "simulation_id": row.simulation_id,
                                        "layout_id": row.layout_id, "family_id": row.family_id,
                                        "wind_direction_deg": row.wind_direction_deg,
                                        "observed": float(observed), "predicted": float(predicted)})
    return pd.DataFrame(prediction_rows), pd.DataFrame(fold_rows), pd.DataFrame(importance_rows)

all_predictions, all_folds, all_importances = [], [], []
for protocol_name in PROTOCOLS:
    print("Running protocol:", protocol_name)
    p, f, imp = run_cv(protocol_name)
    all_predictions.append(p); all_folds.append(f)
    if not imp.empty:
        all_importances.append(imp)

# Feature ablation: compare the original baseline feature set against engineered features under primary validation.
p_base, f_base, _ = run_cv("family_5fold", BASELINE_NUMERIC, CATEGORICAL_FEATURES, feature_set="baseline_features")
all_predictions.append(p_base); all_folds.append(f_base)
predictions = pd.concat(all_predictions, ignore_index=True)
fold_metrics = pd.concat(all_folds, ignore_index=True)
feature_importance_by_fold = pd.concat(all_importances, ignore_index=True) if all_importances else pd.DataFrame()
print("Predictions generated:", len(predictions))
print("Fold metric rows generated:", len(fold_metrics))
display(fold_metrics.head())

## 3. Compute pooled metrics from the fresh predictions

In [ ]:
target_range = float(df[TARGET].max() - df[TARGET].min())
metric_rows = []
for (protocol, feature_set, model_name), group in predictions.groupby(["protocol", "feature_set", "model"], sort=False):
    rmse = float(np.sqrt(mean_squared_error(group.observed, group.predicted)))
    metric_rows.append({
        "protocol": protocol, "feature_set": feature_set, "model": model_name,
        "n_cases": len(group), "n_families": group.family_id.nunique(),
        "mae_m_s": mean_absolute_error(group.observed, group.predicted),
        "rmse_m_s": rmse, "normalized_rmse_target_range": rmse / target_range,
        "r2": r2_score(group.observed, group.predicted),
    })
metrics = pd.DataFrame(metric_rows)
primary_metrics = metrics[(metrics.protocol == "family_5fold") & (metrics.feature_set == "engineered")].sort_values("mae_m_s")
print("Primary result — family-held-out 5-fold CV:")
display(primary_metrics.round(4))
print("All protocols:")
display(metrics.round(4))

## Data-dependent interpretation of the primary results

A report should not merely paste a metric table. This cell identifies the best primary model, quantifies improvement over the mean baseline, compares the family and layout protocols, and checks whether engineered features helped each model. Read these statements alongside the error plots and uncertainty intervals; they are descriptive findings for this dataset, not universal claims.

In [ ]:
primary = metrics[(metrics.protocol == "family_5fold") & (metrics.feature_set == "engineered")].copy()
baseline_row = primary.loc[primary.model == "Mean baseline"].iloc[0]
best_row = primary.loc[primary.model != "Mean baseline"].sort_values("mae_m_s").iloc[0]
print(f"Primary winner by MAE: {best_row.model} (MAE={best_row.mae_m_s:.3f} m/s; RMSE={best_row.rmse_m_s:.3f} m/s; R²={best_row.r2:.3f}).")
print(f"Compared with the mean baseline, the primary winner reduces MAE by {(baseline_row.mae_m_s - best_row.mae_m_s):.3f} m/s ({100*(baseline_row.mae_m_s-best_row.mae_m_s)/baseline_row.mae_m_s:.1f}%).")
layout_rf = metrics[(metrics.protocol == "layout_5fold") & (metrics.feature_set == "engineered") & (metrics.model == "Random Forest")].iloc[0]
family_rf = metrics[(metrics.protocol == "family_5fold") & (metrics.feature_set == "engineered") & (metrics.model == "Random Forest")].iloc[0]
print(f"Random Forest layout-held-out MAE={layout_rf.mae_m_s:.3f} vs family-held-out MAE={family_rf.mae_m_s:.3f} m/s.")
print("If layout-held-out performance is much better, interpret it as evidence that holding out sibling layouts is easier than holding out whole families—not as the primary generalization claim.")
for model_name in ["Ridge Regression", "Random Forest", "XGBoost"]:
    eng = metrics[(metrics.protocol == "family_5fold") & (metrics.feature_set == "engineered") & (metrics.model == model_name)].iloc[0]
    base = metrics[(metrics.protocol == "family_5fold") & (metrics.feature_set == "baseline_features") & (metrics.model == model_name)].iloc[0]
    delta = eng.mae_m_s - base.mae_m_s
    direction = "improved" if delta < 0 else "did not improve" if delta > 0 else "matched"
    print(f"{model_name}: engineered feature set {direction} MAE relative to baseline features by {abs(delta):.3f} m/s.")
print("These conclusions apply to this 224-case scalar dataset and the chosen fixed model settings. They do not demonstrate spatial-field prediction or performance on all urban geometries.")

## 4. Bootstrap uncertainty and paired model comparison

In [ ]:
def family_bootstrap_mae_ci(group, n_boot=1000, seed=42):
    rng = np.random.default_rng(seed)
    family_ids = group.family_id.unique()
    family_errors = {}
    for family in family_ids:
        part = group[group.family_id == family]
        family_errors[family] = (np.abs(part.observed.to_numpy() - part.predicted.to_numpy()).sum(), len(part))
    draws = rng.integers(0, len(family_ids), size=(n_boot, len(family_ids)))
    numerators = np.zeros(n_boot); denominators = np.zeros(n_boot)
    for i in range(n_boot):
        for selected in draws[i]:
            fam = family_ids[selected]
            numerators[i] += family_errors[fam][0]
            denominators[i] += family_errors[fam][1]
    values = numerators / denominators
    return float(np.percentile(values, 2.5)), float(np.percentile(values, 97.5))

ci_rows = []
primary = predictions[(predictions.protocol == "family_5fold") & (predictions.feature_set == "engineered")]
for model_name, group in primary.groupby("model"):
    low, high = family_bootstrap_mae_ci(group)
    ci_rows.append({"model": model_name, "mae_ci95_low": low, "mae_ci95_high": high})
ci_table = pd.DataFrame(ci_rows)
display(ci_table.round(4))

rf = primary[primary.model == "Random Forest"].sort_values("simulation_id")
xgb = primary[primary.model == "XGBoost"].sort_values("simulation_id")
assert rf.simulation_id.tolist() == xgb.simulation_id.tolist()
paired = rf[["simulation_id", "family_id", "observed", "predicted"]].merge(
    xgb[["simulation_id", "predicted"]], on="simulation_id", suffixes=("_rf", "_xgb"), validate="one_to_one")
paired["absolute_error_rf"] = (paired.observed - paired.predicted_rf).abs()
paired["absolute_error_xgb"] = (paired.observed - paired.predicted_xgb).abs()
paired["difference_rf_minus_xgb"] = paired.absolute_error_rf - paired.absolute_error_xgb
# Cluster bootstrap by family for paired MAE difference; positive means XGBoost has lower MAE.
rng = np.random.default_rng(0)
families = paired.family_id.unique()
family_diffs = {fam: paired.loc[paired.family_id == fam, "difference_rf_minus_xgb"].to_numpy() for fam in families}
boot = []
for _ in range(1000):
    selected = rng.choice(families, size=len(families), replace=True)
    errors = np.concatenate([family_diffs[fam] for fam in selected])
    boot.append(errors.mean())
paired_summary = pd.DataFrame([{
    "comparison": "Random Forest MAE minus XGBoost MAE",
    "mean_difference_m_s": float(paired.difference_rf_minus_xgb.mean()),
    "ci95_low": float(np.percentile(boot, 2.5)),
    "ci95_high": float(np.percentile(boot, 97.5)),
    "interpretation": "Positive values favour XGBoost; interval spanning zero means the difference is uncertain."
}])
display(paired_summary.round(4))

## 5. Create comparison and diagnostic plots

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
plot_data = primary_metrics.sort_values("mae_m_s")
ax.bar(plot_data.model, plot_data.mae_m_s)
ax.set(title="Family-held-out model comparison", ylabel="Out-of-fold MAE (m/s)", xlabel="Model")
ax.tick_params(axis="x", rotation=20)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(7, 6))
for model_name, group in primary.groupby("model"):
    ax.scatter(group.observed, group.predicted, alpha=0.65, label=model_name)
low = min(primary.observed.min(), primary.predicted.min())
high = max(primary.observed.max(), primary.predicted.max())
ax.plot([low, high], [low, high], linestyle="--")
ax.set(title="Observed vs out-of-fold predictions", xlabel="Observed wind speed (m/s)", ylabel="Predicted wind speed (m/s)")
ax.legend()
plt.tight_layout()
plt.show()

if not feature_importance_by_fold.empty:
    mean_importance = feature_importance_by_fold.groupby(["model", "feature"], as_index=False).importance.mean()
    for model_name in mean_importance.model.unique():
        top = mean_importance[mean_importance.model == model_name].sort_values("importance", ascending=False).head(10)
        fig, ax = plt.subplots(figsize=(8, 5))
        ax.barh(top.feature[::-1], top.importance[::-1])
        ax.set(title=f"{model_name}: feature importance", xlabel="Mean importance", ylabel="Feature")
        plt.tight_layout()
        plt.show()
else:
    mean_importance = pd.DataFrame()

## 6. Save all results generated in this notebook

In [ ]:
metrics.to_csv(RESULTS / "metrics_all_protocols.csv", index=False)
primary_metrics.to_csv(RESULTS / "model_comparison.csv", index=False)
predictions.to_csv(RESULTS / "oof_predictions.csv", index=False)
fold_metrics.to_csv(RESULTS / "fold_metrics.csv", index=False)
ci_table.to_csv(RESULTS / "family_bootstrap_mae_ci.csv", index=False)
paired_summary.to_csv(RESULTS / "rf_vs_xgb_paired.csv", index=False)
if not mean_importance.empty:
    mean_importance.to_csv(RESULTS / "feature_importance.csv", index=False)

print("Saved freshly computed outputs to", RESULTS)
print("Important: these files were written by the code above; none were read as inputs.")

## How to interpret the final comparison

- Use **family_5fold** as the primary estimate of performance on unseen plan families.
- Treat **family_loo** as a stricter secondary check and **layout_5fold** as an interpolation check; layout-level splits may share related families across train/test.
- Compare the simple mean baseline, Ridge, Random Forest and XGBoost using the same held-out rows.
- A lower MAE/RMSE is better; R² can be negative when the model is worse than predicting the test-fold mean.
- This experiment predicts a single canopy-averaged wind-speed value per case. It is not a temperature model or a spatial-field/U-Net model.